# Data Tour – PLEIAData (main)


## 1. The dataset at a glance

| | |
|---|---|
| **Name** | PLEIAData: consumption, HVAC, temperature, weather and motion sensor data for smart buildings applications |
| **Source** | Zenodo: https://zenodo.org/records/7620136 · paper: Martínez Ibarra, González-Vidal & Skarmeta, *Scientific Data* 10, 118 (2023), DOI 10.1038/s41597-023-02023-3 |
| **System** | The Pleiades building at the University of Murcia (Spain), split into three blocks: **A, B, C** |
| **What is recorded** | Electricity per block, HVAC (heating/cooling units: on/off, mode, setpoint), room temperature, CO₂ and motion in 12 rooms, and outdoor weather from a nearby station |
| **Time** | One year: 1 Jan – 31 Dec 2021 (the processed files stop on **18 Dec 2021**) |
| **Sampling** | Electricity, HVAC, room temperature and weather: every **10 min**; CO₂ every ~2 min; motion as on/off events |
| **Format and size** | CSV files (plus one TXT for weather); zip 283 MB, unzipped ≈ 2.1 GB |
| **Faults** | No labelled faults or failures – normal operation |

## 2. How the folder is organised
**`processed_data`** 
* split per block (A, B, C);
* put on a regular time grid: **`10T` = every 10 minutes, `60T` = every 60 minutes**;
* cleaned (impossible values removed or replaced), extra features added, sources merged;
* all files end on **18 Dec 2021** (where their weather data stopped);
* all timestamps are in **UTC** (Spain = UTC+1 in winter, UTC+2 in summer).

**For this project we plan to work mainly with the processed data**, and to go back to the raw data for quality checks.

## 3. Blocks A, B, C

A, B and C are the **three blocks of one building** (not three buildings). Each block has:
* its **own electricity meter**,
* its own HVAC units,
* its own room temperature sensors.

That is why most processed files exist three times (`…A…`, `…B…`, `…C…`). The **weather files have no A/B/C**: there is one weather station for the whole site.

In [ ]:
import os
import pandas as pd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90)

PROC = "../data/Data_Nature/processed_data/"     # path relative to this notebook

# Column dictionary: what each column means (my understanding, based on the authors' notebook)
W = " (unit not stated in the files; usual unit for this station)"
GLOSSARY = {
    "Date":            "start of the time slot, in UTC",
    "Unnamed: 0":      "row number (no meaning)",
    "dif_cons":        "electricity used by the block IN this time slot (kWh)",
    "dif_cons_real":   "electricity used by the block in this hour (kWh)",
    "dif_cons_smooth": "dif_cons_real smoothed (moving average over ~4 hours) (kWh)",
    "cons_total":      "running total: electricity used since the start of the file (kWh)",
    "V2":              "indoor temperature (°C) – block average, or one room in the room files",
    "V4":              "HVAC on/off – number of units ON in the block (one unit in the room files)",
    "V12":             "HVAC setpoint (°C) – temperature chosen by the user (block average in block files)",
    "V26":             "HVAC unit type: 1 = wall, 2 = ceiling, 0 when OFF (averaged → value between 0 and 2)",
    "V5_0":            "HVAC mode = none (1 = yes, 0 = no)",
    "V5_1":            "HVAC mode = heating (1 = yes, 0 = no)",
    "V5_2":            "HVAC mode = cooling (1 = yes, 0 = no)",
    "V5_4":            "HVAC mode = fan only (1 = yes, 0 = no)",
    "Hour_1":          "time of day 06–14 UTC (1 = yes)",
    "Hour_2":          "time of day 14–22 UTC (1 = yes)",
    "Hour_3":          "time of day 22–06 UTC (1 = yes)",
    "Season_1":        "January–March (1 = yes)",
    "Season_2":        "April–June (1 = yes)",
    "Season_3":        "July–September (1 = yes)",
    "Season_4":        "October–December (1 = yes)",
    "tmed":            "outdoor air temperature (°C)" + W,
    "hrmed":           "outdoor relative humidity (%)" + W,
    "radmed":          "solar radiation (W/m²)" + W,
    "vvmed":           "wind speed (m/s)" + W,
    "dvmed":           "wind direction (degrees)" + W,
    "prec":            "precipitation (mm)" + W,
    "dewpt":           "dew point (°C)" + W,
    "dpv":             "vapour pressure deficit (kPa)" + W,
    "block":           "block of the building (A, B or C)",
    "room":            "room number inside the block",
    "IDhvac":          "ID of the HVAC unit in that room",
    "IDsensor":        "ID of the temperature sensor in that room",
}

def describe(file, nrows=None):
    # Print size and time span of a file and return a table: column | meaning | first value
    df = pd.read_csv(PROC + file, sep=";", nrows=nrows)
    if nrows is None:
        d = pd.to_datetime(df["Date"], utc=True)
        print(f"{file}: {len(df):,} rows × {df.shape[1]} columns   |   {d.min():%d %b %Y %H:%M} → {d.max():%d %b %Y %H:%M} (UTC)")
    else:
        print(f"{file}: {df.shape[1]} columns (only the first {nrows} rows are read)")
    return pd.DataFrame({"column": df.columns,
                         "meaning": [GLOSSARY.get(c, "?") for c in df.columns],
                         "first value": [df[c].iloc[0] for c in df.columns]})

print("Files in processed_data:")
for f in sorted(os.listdir(PROC)):
    print("  ", f)

Files in processed_data:
   consA-10T.csv
   consA-60T.csv
   consB-10T.csv
   consB-60T.csv
   consC-10T.csv
   consC-60T.csv
   data-model-consumoA-60T.csv
   data-model-consumoB-60T.csv
   data-model-consumoC-60T.csv
   data-room10T-all.csv
   data-roomA-10T.csv
   data-roomB-10T.csv
   data-roomC-10T.csv
   data-weather-10T.csv
   data-weather-60T.csv
   hvac-aggA-60T.csv
   hvac-aggB-60T.csv
   hvac-aggC-60T.csv
   temp-sensorA-60T.csv
   temp-sensorB-60T.csv
   temp-sensorC-60T.csv


## 4. Electricity: `consA/B/C-10T.csv` and `consA/B/C-60T.csv`

**This is the target of the project: how much electricity each block uses.**

Each block has **one meter**. The raw meter works like a car's odometer: it only counts up. From it the authors made two numbers per time slot:
* **`dif_cons`** = how much electricity was used **in this slot** (like "how far you drove in this interval")
* **`cons_total`** = **running total** of all slots so far (like "how far since the start")

The **time slot** starts at the timestamp in `Date`:
* `10T` file: row `00:00` = from 00:00 to 00:10
* `60T` file: row `00:00` = from 00:00 to 01:00

The 60T file is the same meter at a coarser zoom: one hour = six 10-min slots added up. In the 60T files the authors also **deleted hours** with `dif_cons` ≤ 0 or ≥ 25 kWh (zeros and spikes around logging gaps) and added a smoothed column.

In [ ]:
describe("consA-10T.csv")

consA-10T.csv: 50,545 rows × 3 columns   |   01 Jan 2021 00:00 → 18 Dec 2021 00:00 (UTC)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:00:00+00:00
1,dif_cons,electricity used by the block IN this time slot (kWh),0.21875
2,cons_total,running total: electricity used since the start of the file (kWh),1.3125


In [ ]:
c10 = pd.read_csv(PROC + "consA-10T.csv", sep=";")
c60 = pd.read_csv(PROC + "consA-60T.csv", sep=";")
print("First 6 rows of the 10-min file (block A):")
print(c10.head(6).to_string(index=False))
print(f"\nSum of these six 10-min values: {c10.dif_cons.head(6).sum():.2f} kWh")
print(f"First row of the 60-min file:   {c60.dif_cons_real.iloc[0]:.2f} kWh   -> same hour, same meter")

First 6 rows of the 10-min file (block A):
                     Date  dif_cons  cons_total
2021-01-01 00:00:00+00:00  0.218750    1.312500
2021-01-01 00:10:00+00:00  0.234375    1.546875
2021-01-01 00:20:00+00:00  0.218750    1.765625
2021-01-01 00:30:00+00:00  0.203125    1.968750
2021-01-01 00:40:00+00:00  0.234375    2.203125
2021-01-01 00:50:00+00:00  0.234375    2.437500

Sum of these six 10-min values: 1.34 kWh
First row of the 60-min file:   1.34 kWh   -> same hour, same meter


In [ ]:
describe("consA-60T.csv")

consA-60T.csv: 8,266 rows × 4 columns   |   01 Jan 2021 00:00 → 18 Dec 2021 00:00 (UTC)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:00:00+00:00
1,dif_cons_real,electricity used by the block in this hour (kWh),1.34
2,cons_total,running total: electricity used since the start of the file (kWh),2.44
3,dif_cons_smooth,dif_cons_real smoothed (moving average over ~4 hours) (kWh),1.25


## 5. HVAC: `hvac-aggA/B/C-60T.csv`

The building is heated and cooled by individual **HVAC units** (split units in the rooms). Per unit the system records whether it is **on or off**, its **mode** (heating / cooling), the **setpoint** (the temperature the user wants) and its **type** (wall or ceiling). The authors summarised all units of a block per hour.

The `V5_*`, `Hour_*` and `Season_*` columns are **dummy columns**: exactly one of each group is 1, the others are 0.

In [ ]:
describe("hvac-aggA-60T.csv")

hvac-aggA-60T.csv: 8,425 rows × 14 columns   |   01 Jan 2021 00:00 → 18 Dec 2021 00:00 (UTC)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:00:00+00:00
1,V4,HVAC on/off – number of units ON in the block (one unit in the room files),1.0
2,V12,HVAC setpoint (°C) – temperature chosen by the user (block average in block files),14.1
3,V26,"HVAC unit type: 1 = wall, 2 = ceiling, 0 when OFF (averaged → value between 0 and 2)",0.0
4,V5_0,"HVAC mode = none (1 = yes, 0 = no)",0
5,V5_1,"HVAC mode = heating (1 = yes, 0 = no)",1
6,V5_2,"HVAC mode = cooling (1 = yes, 0 = no)",0
7,Hour_1,time of day 06–14 UTC (1 = yes),0
8,Hour_2,time of day 14–22 UTC (1 = yes),0
9,Hour_3,time of day 22–06 UTC (1 = yes),1


## 6. Indoor temperature: `temp-sensorA/B/C-60T.csv`

Average room temperature of all sensors in a block, per hour. The authors replaced readings below 15 °C or above 40 °C with the previous valid value.

In [ ]:
describe("temp-sensorA-60T.csv")

temp-sensorA-60T.csv: 8,424 rows × 2 columns   |   01 Jan 2021 01:00 → 18 Dec 2021 00:00 (UTC)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 01:00:00+00:00
1,V2,"indoor temperature (°C) – block average, or one room in the room files",22.686829


## 7. Weather: `data-weather-10T.csv` and `data-weather-60T.csv`

Outdoor weather from one station (MU62), the same for all three blocks – so no A/B/C. Gaps were filled by repeating the last value (forward fill).

In [ ]:
describe("data-weather-60T.csv")

data-weather-60T.csv: 8,424 rows × 9 columns   |   01 Jan 2021 00:00 → 17 Dec 2021 23:00 (UTC)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:00:00+00:00
1,tmed,outdoor air temperature (°C) (unit not stated in the files; usual unit for this station),3.3
2,hrmed,outdoor relative humidity (%) (unit not stated in the files; usual unit for this station),64.4
3,radmed,solar radiation (W/m²) (unit not stated in the files; usual unit for this station),0.0
4,vvmed,wind speed (m/s) (unit not stated in the files; usual unit for this station),0.3
5,dvmed,wind direction (degrees) (unit not stated in the files; usual unit for this station),208.0
6,prec,precipitation (mm) (unit not stated in the files; usual unit for this station),0.0
7,dewpt,dew point (°C) (unit not stated in the files; usual unit for this station),-2.7
8,dpv,vapour pressure deficit (kPa) (unit not stated in the files; usual unit for this station),0.3


In [ ]:
describe("data-weather-10T.csv", nrows=5)

data-weather-10T.csv: 9 columns (only the first 5 rows are read)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:10:00+00:00
1,tmed,outdoor air temperature (°C) (unit not stated in the files; usual unit for this station),3.6
2,hrmed,outdoor relative humidity (%) (unit not stated in the files; usual unit for this station),65.4
3,radmed,solar radiation (W/m²) (unit not stated in the files; usual unit for this station),0.0
4,vvmed,wind speed (m/s) (unit not stated in the files; usual unit for this station),0.6
5,dvmed,wind direction (degrees) (unit not stated in the files; usual unit for this station),180.7
6,prec,precipitation (mm) (unit not stated in the files; usual unit for this station),0.0
7,dewpt,dew point (°C) (unit not stated in the files; usual unit for this station),-2.3
8,dpv,vapour pressure deficit (kPa) (unit not stated in the files; usual unit for this station),0.3


## 8. The merged table per block: `data-model-consumoA/B/C-60T.csv`

The authors' **ready-to-use hourly table** for predicting electricity ("consumo" = consumption): electricity + indoor temperature + HVAC + weather + time features of one block, in one file. **This is probably the main table I will work with.**

Extra steps the authors took here: in block A, hours with ≤ 1 HVAC unit ON were removed; in blocks A and C the number of units ON (`V4`) was smoothed.

In [ ]:
describe("data-model-consumoA-60T.csv")

data-model-consumoA-60T.csv: 7,611 rows × 27 columns   |   05 Jan 2021 07:00 → 17 Dec 2021 23:00 (UTC)


,column,meaning,first value
0,Unnamed: 0,row number (no meaning),0
1,Date,"start of the time slot, in UTC",2021-01-05 07:00:00+00:00
2,dif_cons_real,electricity used by the block in this hour (kWh),7.12
3,cons_total,running total: electricity used since the start of the file (kWh),152.14
4,dif_cons_smooth,dif_cons_real smoothed (moving average over ~4 hours) (kWh),3.63
5,V2,"indoor temperature (°C) – block average, or one room in the room files",21.126829
6,V4,HVAC on/off – number of units ON in the block (one unit in the room files),2.0
7,V12,HVAC setpoint (°C) – temperature chosen by the user (block average in block files),19.24
8,V26,"HVAC unit type: 1 = wall, 2 = ceiling, 0 when OFF (averaged → value between 0 and 2)",0.1
9,V5_0,"HVAC mode = none (1 = yes, 0 = no)",0.0


In [ ]:
for b in "ABC":
    t = pd.read_csv(PROC + f"data-model-consumo{b}-60T.csv", sep=";")
    print(f"Block {b}: {len(t):,} hourly rows")

Block A: 7,611 hourly rows
Block B: 8,266 hourly rows
Block C: 7,963 hourly rows


## 9. Room-level tables: `data-roomA/B/C-10T.csv` and `data-room10T-all.csv`

The same kind of merged table, but **per room and every 10 minutes**, made by the authors for predicting **room temperature**. Each row = one room at one time: its HVAC unit, its temperature sensor, plus the block's electricity and the weather. The `-all` file combines the three blocks and keeps only rooms that have both an HVAC unit and a temperature sensor.

These files are large (42–357 MB; the `-all` file has about 2.5 million rows), so only the first rows are read here.

In [ ]:
describe("data-room10T-all.csv", nrows=5)

data-room10T-all.csv: 27 columns (only the first 5 rows are read)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:10:00+00:00
1,room,room number inside the block,34.0
2,block,"block of the building (A, B or C)",A
3,dif_cons,electricity used by the block IN this time slot (kWh),0.23
4,cons_total,running total: electricity used since the start of the file (kWh),1.55
5,V12,HVAC setpoint (°C) – temperature chosen by the user (block average in block files),21.0
6,V4,HVAC on/off – number of units ON in the block (one unit in the room files),0.0
7,V26,"HVAC unit type: 1 = wall, 2 = ceiling, 0 when OFF (averaged → value between 0 and 2)",0.0
8,V5_0,"HVAC mode = none (1 = yes, 0 = no)",1.0
9,V5_1,"HVAC mode = heating (1 = yes, 0 = no)",0.0


In [ ]:
describe("data-roomB-10T.csv", nrows=5)

data-roomB-10T.csv: 30 columns (only the first 5 rows are read)


,column,meaning,first value
0,Date,"start of the time slot, in UTC",2021-01-01 00:10:00+00:00
1,dif_cons,electricity used by the block IN this time slot (kWh),0.71
2,cons_total,running total: electricity used since the start of the file (kWh),4.8
3,IDhvac,ID of the HVAC unit in that room,335545143
4,block,"block of the building (A, B or C)",B
5,room,room number inside the block,2
6,V12,HVAC setpoint (°C) – temperature chosen by the user (block average in block files),23.0
7,V4,HVAC on/off – number of units ON in the block (one unit in the room files),1.0
8,V26,"HVAC unit type: 1 = wall, 2 = ceiling, 0 when OFF (averaged → value between 0 and 2)",1.0
9,V5_0,"HVAC mode = none (1 = yes, 0 = no)",0


## 10. How we currently see the system (to be checked)

| Role | What I think it is in this data |
|---|---|
| **Output / target y** | electricity per block (`dif_cons` / `dif_cons_real`) |
| **Inputs u (things someone can control)** | HVAC on/off (`V4`), setpoint (`V12`), mode (`V5_*`) |
| **Disturbances d (cannot be controlled)** | outdoor weather (`tmed`, `radmed`, `hrmed`, …); people using the building |
| **States x (the building's condition)** | indoor temperature (`V2`) |
| **Hidden states x (likely there, but not measured)** | heat stored in walls and floors (thermal mass); occupancy (how many people are in the building); condition of the HVAC units (wear, efficiency) |
| **Only partly measured** | occupancy – motion and CO₂ sensors exist for 12 rooms only (raw data) |

A possible purpose: forecast each block's electricity from the weather and the HVAC settings, and later ask "what if" questions about the settings.

## 11. Things we noticed when opening the files (to confirm)

* The processed files end on **18 Dec 2021**, the raw files on 31 Dec.
* All processed timestamps are in **UTC**, so `Hour_*` is not Spanish local time.
* In the raw data, all three electricity meters and the HVAC log have **no readings on 12–15 Aug and 10–13 Dec 2021**.
* In the raw temperature file, **4 sensors report only impossible values**.
* In the raw HVAC file, about **13 % of setpoint values are impossible** (below 10 °C or above 35 °C).
* In the 60T electricity files, hours were **deleted** during cleaning, so the hourly grid has holes.



##. Questions we want to answer with this data

These are open questions – Do not know the answers yet.

1. **Physical system:** which physical components are behind the data (building, HVAC units, meters), and how do they connect?
2. **Inputs → output:** how strongly do weather and HVAC settings affect electricity use? Is outdoor temperature the main driver?
3. **Controllable vs. not controllable:** which variables can actually be changed (setpoint, on/off) and which cannot (weather, people)?
4. **Cause or correlation:** does HVAC drive electricity, or do some variables only move together (e.g. both follow the time of day)?
5. **Hidden or missing states:** what is not measured but would improve the model (occupancy in all rooms, HVAC electricity per unit, the university calendar, heat stored in the walls)?
6. **Time scales:** how fast does electricity react when units switch on? How fast do rooms follow outdoor temperature? How different are day and night, weekdays and weekends, summer and winter?
7. **Enough history:** is one year enough? Every season appears only once – what does that mean for testing a model fairly?
8. **Data quality:** where are the gaps and wrong values, and how did the authors' cleaning change the data?